In [1]:
import matplotlib
matplotlib.use('TkAgg')

import ezpadova
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.widgets import Slider


In [15]:
r = ezpadova.get_isochrones(photsys_file='wfc3_202101_wide', logage=(8, 12, 0.25), MH=(-2,1,0.25))

Querying http://stev.oapd.inaf.it/cgi-bin/cmd...
Retrieving data...


In [3]:
r.columns

Index(['Zini', 'MH', 'logAge', 'Mini', 'int_IMF', 'Mass', 'logL', 'logTe',
       'logg', 'label', 'McoreTP', 'C_O', 'period0', 'period1', 'period2',
       'period3', 'period4', 'pmode', 'Mloss', 'tau1m', 'X', 'Y', 'Xc', 'Xn',
       'Xo', 'Cexcess', 'Z', 'mbolmag', 'F218Wmag', 'F225Wmag', 'F275Wmag',
       'F336Wmag', 'F390Wmag', 'F438Wmag', 'F475Wmag', 'F555Wmag', 'F606Wmag',
       'F625Wmag', 'F775Wmag', 'F814Wmag', 'F105Wmag', 'F110Wmag', 'F125Wmag',
       'F140Wmag', 'F160Wmag'],
      dtype='str')

In [9]:
plt.scatter(r['F606Wmag'] - r['F814Wmag'], r['F606Wmag'])
plt.gca().invert_yaxis()
plt.show()

In [4]:
mystery = pd.read_csv('mystery_gc.dat', names = ['F606W', 'F814W', 'F606W_err', 'F814W_err'], delimiter='\\s+')


In [5]:
plt.scatter(mystery['F606W'] - mystery['F814W'], mystery['F606W'])
plt.gca().invert_yaxis()
plt.show()

In [18]:
def absolute_to_apparent(M606, M814, d_pc, A606=0.06, A814=0.04):
    """
    Convert PARSEC absolute magnitudes to apparent magnitudes.
    (via chatGPT)

    Parameters
    ----------
    M606 : array-like
        Absolute magnitudes in F606W
    M814 : array-like
        Absolute magnitudes in F814W
    d_pc : float
        Distance in parsecs
    A606 : float
        Extinction in F606W (default 0.06)
    A814 : float
        Extinction in F814W (default 0.04)

    Returns
    -------
    m606, m814 : array-like
        Apparent magnitudes
    """

    mu = 5 * np.log10(d_pc) - 5

    m606 = M606 + mu + A606
    m814 = M814 + mu + A814

    return m606, m814

In [7]:
print(np.unique(r['logAge']))
print(np.unique(r['MH']))

[ 8.       8.1      8.2      8.3      8.4      8.5      8.6      8.7
  8.8      8.9      9.       9.1      9.2      9.3      9.40001  9.50001
  9.60001  9.70001  9.80001  9.90001 10.00001 10.10001 10.20001 10.30001
 10.40001 10.50001]
[-1.      -0.75    -0.5     -0.25     0.       0.25     0.5      0.69525]


In [ ]:
for age in np.unique(r['logAge']):
    for metal in np.unique(r['MH']):
        iso = r[(r['logAge'] == age) & (r['MH'] == metal)]
        m606, m814 = absolute_to_apparent(iso['F606Wmag'], iso['F814Wmag'], d_pc=5e5)
        
        plt.scatter(m606 - m814, m606, label = 'Isochrones')
        plt.scatter(mystery['F606W'] - mystery['F814W'], mystery['F606W'], label = 'Mystery Cluster')
        
        plt.gca().invert_yaxis()
        plt.legend()
        plt.xlabel('F606 - F814')
        plt.ylabel('F606')
        plt.title('age = {}, MH = {}'.format(age, metal))
        plt.show()
        plt.close('all')

In [ ]:
# --- initial parameters ---
age0 = np.unique(r['logAge'])[0]
mh0 = np.unique(r['MH'])[0]
d0 = 8000


# --- select starting isochrone ---
iso = r[(r['logAge'] == age0) & (r['MH'] == mh0)]

m606, m814 = absolute_to_apparent(
    iso['F606Wmag'], iso['F814Wmag'], d0
)

# --- plot ---
fig, ax = plt.subplots(figsize=(6,6))
plt.subplots_adjust(left=0.25, bottom=0.3)

cluster = ax.scatter(
    mystery['F606W'] - mystery['F814W'],
    mystery['F606W'],
    s=10,
    label="Cluster"
)

iso_line, = ax.plot(
    m606 - m814,
    m606,
    lw=2,
    color='orange',
    label="Isochrone"
)

ax.invert_yaxis()
ax.set_xlabel("F606W - F814W")
ax.set_ylabel("F606W")
ax.legend()
ax.set_ylim(28.5, 18)
ax.set_xlim(-0.5, 2)


# --- slider axes ---
ax_age = plt.axes([0.25, 0.20, 0.65, 0.03])
ax_mh = plt.axes([0.25, 0.15, 0.65, 0.03])
ax_dist = plt.axes([0.25, 0.10, 0.65, 0.03])

age_slider = Slider(
    ax_age,
    "logAge",
    np.min(r['logAge']),
    np.max(r['logAge']),
    valinit=age0
)

mh_slider = Slider(
    ax_mh,
    "MH",
    np.min(r['MH']),
    np.max(r['MH']),
    valinit=mh0
)

dist_slider = Slider(
    ax_dist,
    "Distance (pc)",
    1000,
    500000,
    valinit=d0
)


# --- update function ---
def update(val):

    age = age_slider.val
    mh = mh_slider.val
    d = dist_slider.val

    # choose closest available isochrone
    age_val = r['logAge'][np.argmin(np.abs(r['logAge'] - age))]
    mh_val = r['MH'][np.argmin(np.abs(r['MH'] - mh))]

    iso = r[(r['logAge'] == age_val) & (r['MH'] == mh_val)]

    m606, m814 = absolute_to_apparent(
        iso['F606Wmag'], iso['F814Wmag'], d
    )

    iso_line.set_xdata(m606 - m814)
    iso_line.set_ydata(m606)

    fig.canvas.draw_idle()


age_slider.on_changed(update)
mh_slider.on_changed(update)
dist_slider.on_changed(update)

plt.show()